# 06 — L-HKS **kernel-times** ablation (MLP3 fixed)

Trains four configs that are **identical** to `configs/GPS/pcqm4m-subset-GPS+LHKS-K16-MLP3.yaml` except `posenc_LHKS.kernel_times` \u2208 **{2, 4, 8, 32}**:

- `pcqm4m-subset-GPS+LHKS-K2-MLP3.yaml`
- `pcqm4m-subset-GPS+LHKS-K4-MLP3.yaml`
- `pcqm4m-subset-GPS+LHKS-K8-MLP3.yaml`
- `pcqm4m-subset-GPS+LHKS-K32-MLP3.yaml`

**W&B:** all runs go to project `pcqm4m-subset-lhks-mlp-ablation`. Auto-generated run names now include the MLP depth, e.g. `...GPS.CustomGatedGCN+Transformer+LHKS-K8-MLP3.r3` (see `graphgps/utils.py::make_wandb_name`).

**Local / Drive:** checkpoints under `results_pcqm4m_subset/mlp_ablation/kernel_times_mlp3/K<K>/seed<seed>/` so each K is a separate subtree (mirrors how `mlp3` / `mlp3_fixed` are organised).

**Prerequisite:** `MyDrive/datasets/pcqm4m-v2/processed/` with the large OGB `.pt` file. Push this repo to GitHub before Colab so the new YAMLs exist after `git clone`.

**Drive account:** **`znidar.mark@gmail.com`** in the Drive auth popup.

**Seeds:** each of the four K configs is trained for **every** seed in `SEEDS` (default **three** seeds: `3, 4, 5`, matching `01_run_mlp3.ipynb`). That is **4 × 3 = 12** training runs total unless you change the lists.

**Time budget:** ~2 h per (K, seed) on A100, ~5 h on T4 — scale by 12 runs for wall-clock planning.

## 1. Configuration — edit seeds and which K values to run

In [ ]:
# =========================================================
# EDIT: exactly which seeds (default: 3 seeds) and which K values
# =========================================================
SEEDS          = [3, 4, 5]       # 3 seeds per K — same default as 01_run_mlp3.ipynb
KERNEL_TIMES   = [2, 4, 8, 32] # each needs configs/GPS/pcqm4m-subset-GPS+LHKS-K{K}-MLP3.yaml
# =========================================================

GITHUB_REPO     = "https://github.com/mark-znidar/gdl__2.git"
DRIVE_WORKSPACE = "/content/drive/MyDrive"

DRIVE_RESULTS  = f"{DRIVE_WORKSPACE}/results_pcqm4m_subset"
DRIVE_DATASETS = f"{DRIVE_WORKSPACE}/datasets"
print("Seeds (each K is run once per seed):", SEEDS)
print("Kernel times K:", KERNEL_TIMES)
print(f"Total runs: {len(KERNEL_TIMES)} K × {len(SEEDS)} seeds = {len(KERNEL_TIMES) * len(SEEDS)}")

## 2. GPU check

Prefer A100 or V100; T4 works but is slower.

In [ ]:
!nvidia-smi | head -20

## 3. Mount Drive

In [ ]:
from google.colab import drive
import os, shutil, time

MOUNTPOINT = "/content/drive"
MOUNT_TIMEOUT_MS = 180_000

def _clean_mountpoint():
    try:
        drive.flush_and_unmount()
    except Exception:
        pass
    if os.path.islink(MOUNTPOINT):
        os.remove(MOUNTPOINT)
    elif os.path.isdir(MOUNTPOINT) and os.listdir(MOUNTPOINT):
        shutil.rmtree(MOUNTPOINT)
    os.makedirs(MOUNTPOINT, exist_ok=True)

def _mount(force_remount=False):
    drive.mount(MOUNTPOINT, timeout_ms=MOUNT_TIMEOUT_MS, force_remount=force_remount)

_clean_mountpoint()
try:
    _mount()
except ValueError as e:
    if "mount failed" not in str(e).lower():
        raise
    print("Drive mount failed — retrying with force_remount in 5s ...", flush=True)
    time.sleep(5)
    _clean_mountpoint()
    _mount(force_remount=True)


## 4. Clone repo + install deps + symlink to Drive

In [ ]:
import os
%cd /content
if not os.path.isdir("gdl__2"):
    !git clone {GITHUB_REPO}
else:
    %cd gdl__2 && !git pull && %cd ..
%cd gdl__2
!bash run_colab/setup.sh
!rm -rf results_pcqm4m_subset datasets
!ln -sfn {DRIVE_RESULTS}  results_pcqm4m_subset
!ln -sfn {DRIVE_DATASETS} datasets
!ls -la results_pcqm4m_subset datasets

## 5. WandB login

In [ ]:
import wandb
wandb.login()

## 6. Sanity check — dataset on Drive + YAMLs present

In [ ]:
from pathlib import Path

proc = Path(DRIVE_DATASETS) / "pcqm4m-v2" / "processed"
assert proc.is_dir(), f"Missing {proc}"
pts = sorted(p for p in proc.glob("*.pt") if p.is_file())
assert pts, f"No .pt under {proc}"
big = [p for p in pts if p.stat().st_size > 500_000_000]
assert big, "No large processed .pt (OGB) found"
for p in big:
    print(f"OK  {p.name}  {p.stat().st_size / 1e9:.1f} GB")

for K in KERNEL_TIMES:
    cfg = Path(f"configs/GPS/pcqm4m-subset-GPS+LHKS-K{K}-MLP3.yaml")
    assert cfg.is_file(), f"Missing {cfg} — git pull / push repo with new configs"
print("All KERNEL_TIMES YAMLs found.")

## 7. Train each (K, seed)

Nested loop: for **each** `K` in `KERNEL_TIMES`, runs **every** seed in `SEEDS` (default 3 seeds). Skips any `(K, seed)` that already has a `.ckpt` under the corresponding Drive path.

In [ ]:
import os, glob, datetime, subprocess

for K in KERNEL_TIMES:
    cfg = f"configs/GPS/pcqm4m-subset-GPS+LHKS-K{K}-MLP3.yaml"
    outbase = f"results_pcqm4m_subset/mlp_ablation/kernel_times_mlp3/K{K}"
    for seed in SEEDS:
        outdir = f"{outbase}/seed{seed}"
        if glob.glob(f"{outdir}/*/ckpt/*.ckpt"):
            print(f"[skip] K={K} seed={seed} — ckpt exists at {outdir}")
            continue
        os.makedirs(outdir, exist_ok=True)
        print(f"\n{'='*60}")
        print(f">>> K={K}  seed={seed}  start {datetime.datetime.now().isoformat(timespec='seconds')}")
        print(f">>> cfg:    {cfg}")
        print(f">>> outdir: {outdir}")
        print(f"{'='*60}", flush=True)
        ret = subprocess.call(["python", "main.py", "--cfg", cfg,
                               "--repeat", "1", "seed", str(seed),
                               "out_dir", outdir])
        if ret != 0:
            raise RuntimeError(f"Training failed K={K} seed={seed} (exit {ret})")
        print(f">>> K={K} seed={seed}  done")
print("\n=== All (K, seed) runs complete ===")

## 8. Verify checkpoints on Drive

In [ ]:
import subprocess, os
root = f"{DRIVE_RESULTS}/mlp_ablation/kernel_times_mlp3"
out = subprocess.check_output(["find", root, "-name", "*.ckpt"], text=True)
paths = sorted(p for p in out.strip().split("\n") if p)
print(f"Found {len(paths)} checkpoint(s) under {root}:")
for p in paths[:40]:
    print(f"  {os.path.getsize(p)/1e6:7.1f} MB  {p}")
if len(paths) > 40:
    print(f"  ... and {len(paths) - 40} more")